# Consulta de Wikidata desde Python usando SPARQL

Este notebook muestra un ejemplo sencillo de cómo consultar **Wikidata Query Service** desde Python.

La idea no es reemplazar SPARQL por Python. Python se utiliza únicamente para enviar la consulta al endpoint y recibir la respuesta en formato JSON.

El flujo es:

```text
Python
  ↓ HTTP
Wikidata Query Service
  ↓
SPARQL
  ↓
Resultados JSON
```

En el taller, esta misma idea puede utilizarse para buscar candidatos de reconciliación para los artistas del grafo local.


## 1. Importar la librería necesaria

Usaremos `requests` para enviar consultas HTTP al endpoint SPARQL de Wikidata.

Si no la tiene instalada, puede ejecutar:

```python
!pip install requests
```


In [ ]:
import requests


## 2. Función genérica para ejecutar consultas SPARQL

Esta función es **genérica**: recibe cualquier consulta SPARQL como texto y devuelve la respuesta de Wikidata en formato JSON.

No es necesario modificarla para cada consulta. Lo que cambia normalmente es la consulta SPARQL que se le pasa como argumento.


In [ ]:
WIKIDATA_ENDPOINT = "https://query.wikidata.org/sparql"

def ejecutar_consulta(query):
    """
    Ejecuta una consulta SPARQL contra Wikidata Query Service.

    Parámetros
    ----------
    query : str
        Consulta SPARQL que se desea ejecutar.

    Retorna
    -------
    dict
        Respuesta de Wikidata en formato JSON.
    """

    response = requests.get(
        WIKIDATA_ENDPOINT,
        params={
            "query": query,
            "format": "json"
        },
        headers={
            "User-Agent": "WebSemantica-Uniandes/1.0"
        },
        timeout=30
    )

    response.raise_for_status()

    return response.json()


## 3. Buscar candidatos por nombre

Suponga que en el grafo local aparece un artista con el nombre:

```text
Frida Kahlo
```

Todavía no conocemos su identificador en Wikidata.

La siguiente consulta utiliza el servicio de búsqueda de Wikidata para recuperar posibles candidatos y, para cada uno, intenta obtener algunas propiedades útiles para compararlos:

- nombre
- fecha de nacimiento
- país de ciudadanía

Estas propiedades pueden funcionar como **señales de reconciliación**.


In [ ]:
nombre_artista = "Frida Kahlo"

# Skip caracteres que podrían interferir con la consulta.
nombre_seguro = nombre_artista.replace("\\", "\\\\").replace('"', '\\"')

query = f"""
PREFIX bd: <http://www.bigdata.com/rdf#>
PREFIX wikibase: <http://wikiba.se/ontology#>
PREFIX mwapi: <https://www.mediawiki.org/ontology#API/>
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>

SELECT DISTINCT
    ?item
    ?itemLabel
    ?birthDate
    ?country
    ?countryLabel
WHERE {{

    # Busca entidades cuyo nombre se parezca al nombre del artista.
    SERVICE wikibase:mwapi {{
        bd:serviceParam
            wikibase:endpoint "www.wikidata.org" ;
            wikibase:api "EntitySearch" ;
            mwapi:search "{nombre_seguro}" ;
            mwapi:language "en" .

        #  Guarda en ?item cada entidad encontrada por EntitySearch.
        ?item wikibase:apiOutputItem mwapi:item .

        # Guarda el orden de relevancia entregado por el buscador.
        ?num wikibase:apiOrdinal true .
    }}

    # P31 = instance of
    # Q5  = human
    # Para este ejemplo nos interesan únicamente personas.
    ?item wdt:P31 wd:Q5 .

    # P569 = date of birth
    OPTIONAL {{
        ?item wdt:P569 ?birthDate .
    }}

    # P27 = country of citizenship
    OPTIONAL {{
        ?item wdt:P27 ?country .
    }}

    # Obtiene etiquetas legibles para los recursos.
    SERVICE wikibase:label {{
        bd:serviceParam wikibase:language "es,en" .
    }}
}}
ORDER BY ASC(?num)
LIMIT 10
"""

resultado = ejecutar_consulta(query)


## 4. Inspeccionar la respuesta

La respuesta completa es un diccionario de Python con la estructura JSON retornada por Wikidata.

Los resultados de la consulta se encuentran en:

```python
resultado["results"]["bindings"]
```

Cada elemento corresponde a una fila de la consulta SPARQL.


In [6]:
resultado["results"]["bindings"][:2]


[{'item': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q5588'},
  'itemLabel': {'xml:lang': 'es', 'type': 'literal', 'value': 'Frida Kahlo'},
  'birthDate': {'datatype': 'http://www.w3.org/2001/XMLSchema#dateTime',
   'type': 'literal',
   'value': '1907-07-06T00:00:00Z'},
  'country': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q96'},
  'countryLabel': {'xml:lang': 'es', 'type': 'literal', 'value': 'México'}}]

Podemos recorrer los resultados y mostrar únicamente los campos que nos interesan.


In [7]:
for fila in resultado["results"]["bindings"]:
    print("Recurso:", fila["item"]["value"])
    print("Nombre:", fila.get("itemLabel", {}).get("value"))
    print("Nacimiento:", fila.get("birthDate", {}).get("value"))
    print("País:", fila.get("countryLabel", {}).get("value"))
    print("-" * 50)


Recurso: http://www.wikidata.org/entity/Q5588
Nombre: Frida Kahlo
Nacimiento: 1907-07-06T00:00:00Z
País: México
--------------------------------------------------


## 5. Convertir los resultados a una estructura más sencilla

Para trabajar con varios artistas puede resultar más cómodo transformar la respuesta de Wikidata en una lista de diccionarios.

Aquí extraemos:

- `qid`: identificador de Wikidata;
- `nombre`;
- `nacimiento`;
- `pais`.

**Importante:** esta función solamente organiza los candidatos encontrados.  
No decide cuál candidato corresponde al artista del grafo local.


In [8]:
def extraer_candidatos(resultado):
    """
    Convierte la respuesta JSON de Wikidata en una lista sencilla
    de candidatos de reconciliación.
    """

    candidatos = []

    for fila in resultado["results"]["bindings"]:
        uri = fila["item"]["value"]

        # El QID es la última parte de la URI:
        # http://www.wikidata.org/entity/QXXXX -> QXXXX
        qid = uri.rsplit("/", 1)[-1]

        candidatos.append({
            "qid": qid,
            "nombre": fila.get("itemLabel", {}).get("value"),
            "nacimiento": fila.get("birthDate", {}).get("value"),
            "pais": fila.get("countryLabel", {}).get("value")
        })

    return candidatos

candidatos = extraer_candidatos(resultado)
candidatos


[{'qid': 'Q5588',
  'nombre': 'Frida Kahlo',
  'nacimiento': '1907-07-06T00:00:00Z',
  'pais': 'México'}]

[texto del enlace](https://)## 7. Siguiente paso en el taller

Una vez aceptada una correspondencia, el recurso local puede enlazarse con Wikidata.

Por ejemplo:

```turtle
<http://example.org/moma/artist/123>
    owl:sameAs
    <http://www.wikidata.org/entity/QXXXX> .
```

Posteriormente, este enlace puede utilizarse para combinar información del grafo local con información que solamente está disponible en Wikidata.

artista_local = URIRef("http://example.org/moma/artist/123")
artista_wikidata = URIRef("http://www.wikidata.org/entity/QXXXX")

### Pseudocódigo:

```python
grafo.add((
    artista_local,
    OWL.sameAs,
    artista_wikidata
))
```

## **Desarrollo del Taller**

## PARTE UNO — Construcción del grafo local

### Punto 1 - Definición de una política de IRI
Se propone utilizar la estructura http://example.org/moma/artist/{ConstituentID} para identificar a cada artista del conjunto de datos. Para construir estos IRI se utilizará la columna ConstituentID, que contiene el identificador numérico del artista.

Por ejemplo, para el registro de Robert Arneson, cuyo ConstituentID es 1, el IRI correspondiente será http://example.org/moma/artist/1.

Como decisión no utilizaré directamente DisplayName como identificador, porque pueden existir artistas con nombres iguales, variaciones en la escritura o cambios en los nombres. Además, los espacios y caracteres especiales requieren un tratamiento adecuado. Utilizar ConstituentID permite obtener identificadores más estables y evitar ambigüedades, siempre que los valores sean únicos y persistentes en el conjunto de datos.

### Punto 2 - Generación del grafo RDF

In [9]:
# Instalar librerias
!pip install rdflib pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 6.3 MB/s  0:00:00


In [11]:
# Cargar archivo de artistas
import pandas as pd

filename = "/content/moma-artists.txt"

df = pd.read_csv(filename, sep=None, engine="python")

print("Columnas:", df.columns.tolist())
print("Número de registros:", len(df))
display(df.head())

Columnas: ['ConstituentID', 'DisplayName', 'ArtistBio', 'Nationality', 'Gender', 'BeginDate', 'EndDate']
Número de registros: 90


,ConstituentID,DisplayName,ArtistBio,Nationality,Gender,BeginDate,EndDate
0,1,Robert Arneson,"American, 1930–1992",American,male,1930,1992
1,2,Doroteo Arnaiz,"Spanish, born 1936",Spanish,male,1936,0
2,3,Bill Arnold,"American, born 1941",American,male,1941,0
3,4,Charles Arnoldi,"American, born 1946",American,male,1946,0
4,5,Per Arnoldi,"Danish, born 1941",Danish,male,1941,0


In [12]:
# Generar el grafo RDF
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, XSD

# Crear el grafo
g = Graph()

# Definir namespaces
MOMA = Namespace("http://example.org/moma/")
SCHEMA = Namespace("https://schema.org/")
ARTIST = Namespace("http://example.org/moma/ontology#")

g.bind("moma", MOMA)
g.bind("schema", SCHEMA)
g.bind("artist", ARTIST)

# Crear un recurso RDF por artista
for _, row in df.iterrows():

    if pd.isna(row["ConstituentID"]):
        continue

    artist_id = str(row["ConstituentID"]).strip()

    if artist_id.endswith(".0"):
        artist_id = artist_id[:-2]

    artist_uri = URIRef(
        f"http://example.org/moma/artist/{artist_id}"
    )

    # Tipo del recurso
    g.add((artist_uri, RDF.type, MOMA.Artist))

    # Propiedades descriptivas
    properties = {
        "DisplayName": SCHEMA.name,
        "ArtistBio": ARTIST.artistBio,
        "Nationality": ARTIST.nationality,
        "Gender": ARTIST.gender,
        "BeginDate": ARTIST.beginDate,
        "EndDate": ARTIST.endDate
    }

    for column, predicate in properties.items():
        value = row[column]

        if pd.isna(value):
            continue

        if column in ["BeginDate", "EndDate"]:
            try:
                value = int(value)
            except (ValueError, TypeError, OverflowError):
                continue

            obj = Literal(value, datatype=XSD.integer)
        else:
            obj = Literal(str(value).strip())

        g.add((artist_uri, predicate, obj))

print("Grafo generado correctamente.")
print("Total de triples:", len(g))

Grafo generado correctamente.
Total de triples: 623


In [13]:
# Comprobar un artista
artist_uri = URIRef("http://example.org/moma/artist/1")

for predicate, obj in g.predicate_objects(artist_uri):
    print(predicate, "->", obj)

http://www.w3.org/1999/02/22-rdf-syntax-ns#type -> http://example.org/moma/Artist
https://schema.org/name -> Robert Arneson
http://example.org/moma/ontology#artistBio -> American, 1930–1992
http://example.org/moma/ontology#nationality -> American
http://example.org/moma/ontology#gender -> male
http://example.org/moma/ontology#beginDate -> 1930
http://example.org/moma/ontology#endDate -> 1992


In [14]:
# Grafo en turtle
g.serialize(
    destination="/content/artistas.ttl",
    format="turtle"
)

print("Archivo generado en /content/artistas.ttl")

Archivo generado en /content/artistas.ttl


## PARTE DOS — Consulta de Wikidata

In [15]:
# Consulta programática de Wikidata
query_frida = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX wikibase: <http://wikiba.se/ontology#>
PREFIX bd: <http://www.bigdata.com/rdf#>

SELECT ?propiedad ?propiedadLabel ?valor ?valorLabel
WHERE {
    wd:Q5588 ?relacion ?valor .

    VALUES (?relacion ?propiedad) {
        (wdt:P31  wd:P31)
        (wdt:P569 wd:P569)
        (wdt:P570 wd:P570)
        (wdt:P27  wd:P27)
        (wdt:P21  wd:P21)
        (wdt:P106 wd:P106)
    }

    SERVICE wikibase:label {
        bd:serviceParam wikibase:language "es,en" .
    }

    BIND(?propiedad AS ?propiedadLabel)
}
LIMIT 50
"""

resultado_frida = ejecutar_consulta(query_frida)

for fila in resultado_frida["results"]["bindings"]:
    print("Propiedad:", fila.get("propiedadLabel", {}).get("value"))
    print("Valor:", fila.get("valorLabel", {}).get("value",
                                      fila.get("valor", {}).get("value")))
    print("-" * 50)

Propiedad: http://www.wikidata.org/entity/P21
Valor: femenino
--------------------------------------------------
Propiedad: http://www.wikidata.org/entity/P27
Valor: México
--------------------------------------------------
Propiedad: http://www.wikidata.org/entity/P31
Valor: ser humano
--------------------------------------------------
Propiedad: http://www.wikidata.org/entity/P106
Valor: pintor
--------------------------------------------------
Propiedad: http://www.wikidata.org/entity/P569
Valor: 1907-07-06T00:00:00Z
--------------------------------------------------
Propiedad: http://www.wikidata.org/entity/P570
Valor: 1954-07-13T00:00:00Z
--------------------------------------------------


## PARTE TRES — Reconciliación de entidades

In [16]:
# Ampliar consulta para comparar por mas propiedades
def buscar_candidatos(nombre_artista):
    nombre_seguro = (
        nombre_artista
        .replace("\\", "\\\\")
        .replace('"', '\\"')
    )

    query = f"""
    PREFIX bd: <http://www.bigdata.com/rdf#>
    PREFIX wikibase: <http://wikiba.se/ontology#>
    PREFIX mwapi: <https://www.mediawiki.org/ontology#API/>
    PREFIX wd: <http://www.wikidata.org/entity/>
    PREFIX wdt: <http://www.wikidata.org/prop/direct/>

    SELECT DISTINCT
        ?item
        ?itemLabel
        ?birthDate
        ?deathDate
        ?countryLabel
        ?genderLabel
    WHERE {{
        SERVICE wikibase:mwapi {{
            bd:serviceParam
                wikibase:endpoint "www.wikidata.org" ;
                wikibase:api "EntitySearch" ;
                mwapi:search "{nombre_seguro}" ;
                mwapi:language "en" .

            ?item wikibase:apiOutputItem mwapi:item .
            ?num wikibase:apiOrdinal true .
        }}

        # Restringir los candidatos a personas
        ?item wdt:P31 wd:Q5 .

        # Fecha de nacimiento
        OPTIONAL {{ ?item wdt:P569 ?birthDate . }}

        # Fecha de fallecimiento
        OPTIONAL {{ ?item wdt:P570 ?deathDate . }}

        # País de ciudadanía
        OPTIONAL {{ ?item wdt:P27 ?country . }}

        # Género
        OPTIONAL {{ ?item wdt:P21 ?gender . }}

        SERVICE wikibase:label {{
            bd:serviceParam wikibase:language "es,en" .
        }}
    }}
    ORDER BY ASC(?num)
    LIMIT 10
    """

    resultado = ejecutar_consulta(query)
    candidatos = []

    for fila in resultado["results"]["bindings"]:
        uri = fila["item"]["value"]

        candidatos.append({
            "qid": uri.rsplit("/", 1)[-1],
            "nombre_wikidata": fila.get("itemLabel", {}).get("value"),
            "nacimiento": fila.get("birthDate", {}).get("value"),
            "fallecimiento": fila.get("deathDate", {}).get("value"),
            "pais": fila.get("countryLabel", {}).get("value"),
            "genero": fila.get("genderLabel", {}).get("value")
        })

    return candidatos

In [17]:
# Probar con Frida
candidatos_frida = buscar_candidatos("Frida Kahlo")

import pandas as pd

pd.DataFrame(candidatos_frida)

,qid,nombre_wikidata,nacimiento,fallecimiento,pais,genero
0,Q5588,Frida Kahlo,1907-07-06T00:00:00Z,1954-07-13T00:00:00Z,México,femenino


In [21]:
# Probar con los artistas de MoMA
import time

resultados = []

muestra = df.dropna(subset=["DisplayName"]).head(100)

for _, artista in muestra.iterrows():
    nombre = str(artista["DisplayName"]).strip()

    print("Buscando:", nombre)

    try:
        candidatos = buscar_candidatos(nombre)

        for candidato in candidatos:
            resultados.append({
                "nombre_moma": nombre,
                "ConstituentID": artista["ConstituentID"],
                **candidato
            })

    except Exception as e:
        print("Error buscando", nombre, ":", e)

    time.sleep(1)

df_candidatos = pd.DataFrame(resultados)
display(df_candidatos)

Buscando: Robert Arneson
Buscando: Doroteo Arnaiz
Buscando: Bill Arnold
Buscando: Charles Arnoldi
Buscando: Per Arnoldi
Buscando: Danilo Aroldi
Buscando: Bill Aron
Buscando: David Aronson
Buscando: Irene Aronson
Buscando: Jean (Hans) Arp
Buscando: Jüri Arrak
Buscando: José Arellano Fischer
Buscando: Folke Arstrom
Buscando: Cristobal Arteche
Buscando: Artko
Buscando: Richard Artschwager
Buscando: Ruth Asawa
Buscando: Isidora Aschheim
Buscando: Charles Robert Ashbee
Buscando: Donald Ashcraft
Buscando: E. M. Ashe
Buscando: Göran Åslin
Buscando: Erik Gunnar Asplund
Buscando: Geneviève Asse
Buscando: Sergio Asti
Buscando: Dana Atchley
Buscando: Atelier Eggers
Buscando: A.A.P.
Buscando: Alvar Aalto
Buscando: Aino Aalto
Buscando: Eero Aarnio
Buscando: Jules Aarons
Buscando: Magdalena Abakanowicz
Buscando: James Abbe
Buscando: Jussuf Abbo
Buscando: Berenice Abbott
Buscando: Nobuya Abe
Buscando: Abe Shiro (pen Name: Suichiku)
Buscando: Robert Abel
Buscando: Sigmund Abeles
Buscando: Raimund Abra

,nombre_moma,ConstituentID,qid,nombre_wikidata,nacimiento,fallecimiento,pais,genero
0,Robert Arneson,1,Q7341594,Robert Arneson,1930-09-04T00:00:00Z,1992-11-02T00:00:00Z,Estados Unidos,masculino
1,Doroteo Arnaiz,2,Q55996241,Doroteo Arnáiz,1936-01-01T00:00:00Z,None,España,masculino
2,Bill Arnold,3,Q42943193,William A. Arnold,None,None,Estados Unidos,masculino
3,Bill Arnold,3,Q11854243,Bill Arnold,1992-05-13T00:00:00Z,None,Estados Unidos,masculino
4,Bill Arnold,3,Q63303031,Bill Arnold,None,None,Estados Unidos,masculino
...,...,...,...,...,...,...,...,...
185,Joan Albert,99,Q38545039,Joan Albert Barberà,None,None,None,femenino
186,Joan Albert,99,Q102404450,Joan Albert López Vallverdú,None,None,None,masculino
187,Joan Albert,99,Q47274420,Joan-Albert Sanchez-Cabeza,None,None,None,masculino
188,Joan Albert,99,Q63119722,Joan Albert,1943-01-01T00:00:00Z,2012-01-01T00:00:00Z,Estados Unidos,femenino


In [22]:
# Registrar las señales y la decisión
df_candidatos["senales_comparacion"] = ""
df_candidatos["decision"] = "Pendiente de revisión"

columnas_finales = [
    "nombre_moma",
    "ConstituentID",
    "qid",
    "nombre_wikidata",
    "nacimiento",
    "fallecimiento",
    "pais",
    "genero",
    "senales_comparacion",
    "decision"
]

df_revision = df_candidatos[columnas_finales].copy()

display(df_revision)

,nombre_moma,ConstituentID,qid,nombre_wikidata,nacimiento,fallecimiento,pais,genero,senales_comparacion,decision
0,Robert Arneson,1,Q7341594,Robert Arneson,1930-09-04T00:00:00Z,1992-11-02T00:00:00Z,Estados Unidos,masculino,,Pendiente de revisión
1,Doroteo Arnaiz,2,Q55996241,Doroteo Arnáiz,1936-01-01T00:00:00Z,None,España,masculino,,Pendiente de revisión
2,Bill Arnold,3,Q42943193,William A. Arnold,None,None,Estados Unidos,masculino,,Pendiente de revisión
3,Bill Arnold,3,Q11854243,Bill Arnold,1992-05-13T00:00:00Z,None,Estados Unidos,masculino,,Pendiente de revisión
4,Bill Arnold,3,Q63303031,Bill Arnold,None,None,Estados Unidos,masculino,,Pendiente de revisión
...,...,...,...,...,...,...,...,...,...,...
185,Joan Albert,99,Q38545039,Joan Albert Barberà,None,None,None,femenino,,Pendiente de revisión
186,Joan Albert,99,Q102404450,Joan Albert López Vallverdú,None,None,None,masculino,,Pendiente de revisión
187,Joan Albert,99,Q47274420,Joan-Albert Sanchez-Cabeza,None,None,None,masculino,,Pendiente de revisión
188,Joan Albert,99,Q63119722,Joan Albert,1943-01-01T00:00:00Z,2012-01-01T00:00:00Z,Estados Unidos,femenino,,Pendiente de revisión


In [24]:
# Exportar los resultados a CSV
df_revision.to_csv(
    "reconciliacion_candidatos.csv",
    index=False,
    encoding="utf-8-sig"
)

from google.colab import files
files.download("reconciliacion_candidatos.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### PARTE CUATRO — Enlazado de entidades

In [25]:
# Cargar el CSV y preparar los enlaces
import pandas as pd

df_revision = pd.read_csv(
    "/content/reconciliacion_candidatos.csv"
)

# Agregar la columna si todavía no existe
if "tipo_enlace" not in df_revision.columns:
    df_revision["tipo_enlace"] = ""

display(df_revision.head())

,nombre_moma,ConstituentID,qid,nombre_wikidata,nacimiento,fallecimiento,pais,genero,senales_comparacion,decision,tipo_enlace
0,Robert Arneson,1,Q7341594,Robert Arneson,1930-09-04T00:00:00Z,1992-11-02T00:00:00Z,Estados Unidos,masculino,NaN,Pendiente de revisión,
1,Doroteo Arnaiz,2,Q55996241,Doroteo Arnáiz,1936-01-01T00:00:00Z,NaN,España,masculino,NaN,Pendiente de revisión,
2,Bill Arnold,3,Q42943193,William A. Arnold,NaN,NaN,Estados Unidos,masculino,NaN,Pendiente de revisión,
3,Bill Arnold,3,Q11854243,Bill Arnold,1992-05-13T00:00:00Z,NaN,Estados Unidos,masculino,NaN,Pendiente de revisión,
4,Bill Arnold,3,Q63303031,Bill Arnold,NaN,NaN,Estados Unidos,masculino,NaN,Pendiente de revisión,


In [28]:
# Mostrar los primeros 30 candidatos
display(
    df_revision[
        [
            "ConstituentID",
            "nombre_moma",
            "qid",
            "nombre_wikidata",
            "nacimiento",
            "fallecimiento",
            "pais",
            "genero"
        ]
    ].head(30)
)

,ConstituentID,nombre_moma,qid,nombre_wikidata,nacimiento,fallecimiento,pais,genero
0,1,Robert Arneson,Q7341594,Robert Arneson,1930-09-04T00:00:00Z,1992-11-02T00:00:00Z,Estados Unidos,masculino
1,2,Doroteo Arnaiz,Q55996241,Doroteo Arnáiz,1936-01-01T00:00:00Z,NaN,España,masculino
2,3,Bill Arnold,Q42943193,William A. Arnold,NaN,NaN,Estados Unidos,masculino
3,3,Bill Arnold,Q11854243,Bill Arnold,1992-05-13T00:00:00Z,NaN,Estados Unidos,masculino
4,3,Bill Arnold,Q63303031,Bill Arnold,NaN,NaN,Estados Unidos,masculino
5,3,Bill Arnold,Q4907946,Bill Arnold,NaN,NaN,NaN,masculino
6,3,Bill Arnold,Q77287962,Q77287962,1941-01-01T00:00:00Z,NaN,Estados Unidos,masculino
7,4,Charles Arnoldi,Q1063584,Charles Arnoldi,1946-04-10T00:00:00Z,NaN,Estados Unidos,masculino
8,4,Charles Arnoldi,Q55208542,Charles Arnoldi,1779-01-01T00:00:00Z,1817-01-01T00:00:00Z,Bajo Canadá,masculino
9,5,Per Arnoldi,Q3373273,Per Arnoldi,1941-05-25T00:00:00Z,NaN,Reino de Dinamarca,masculino


In [29]:

import pandas as pd
import numpy as np
import unicodedata
import re

# ============================================================
# REGLAS DE RECONCILIACIÓN DE ENTIDADES
# ============================================================
#
# OBJETIVO:
# Identificar automáticamente qué candidato de Wikidata
# corresponde a cada artista del dataset de MoMA.
#
# REGLAS DE PUNTUACIÓN:
#
# 1. Nombre:
#    - Coincidencia exacta normalizada: +3 puntos.
#    - Coincidencia parcial: +1 punto.
#
# 2. Año de nacimiento:
#    - Coincidencia: +4 puntos.
#    - Diferencia: -5 puntos.
#
# 3. Año de fallecimiento:
#    - Coincidencia: +4 puntos.
#    - Diferencia: -5 puntos.
#
# 4. Género:
#    - Coincidencia: +2 puntos.
#    - Diferencia: -4 puntos.
#
# 5. Nacionalidad / país:
#    - Coincidencia aproximada: +1 punto.
#    - No se penaliza una diferencia porque nacionalidad
#      y ciudadanía no siempre representan lo mismo.
#
# DECISIONES:
#
# ACEPTAR:
#    - Puntuación >= 10.
#    - Al menos dos señales coincidentes.
#    - Sin contradicciones en nacimiento, fallecimiento
#      o género.
#
# RECHAZAR:
#    - Puntuación <= 0.
#    - O contradicción en nacimiento o fallecimiento
#      cuando ambas fuentes tienen el dato.
#
# PENDIENTE DE REVISIÓN:
#    - Cualquier otro caso.
#
# IMPORTANTE:
# Estas reglas son heurísticas académicas.
# Una decisión automática no constituye una prueba absoluta
# de identidad. Los casos aceptados deben poder justificarse
# con la evidencia registrada.
# ============================================================


# ------------------------------------------------------------
# 1. Normalización de textos
# ------------------------------------------------------------

def normalizar(valor):

    if pd.isna(valor):
        return ""

    texto = str(valor).strip().lower()

    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(
        c for c in texto
        if not unicodedata.combining(c)
    )

    texto = re.sub(r"[^a-z0-9 ]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()

    return texto


# ------------------------------------------------------------
# 2. Obtener año de fechas
# ------------------------------------------------------------

def obtener_anio(valor):

    if pd.isna(valor):
        return None

    try:
        return int(str(valor)[:4])
    except:
        return None


# ------------------------------------------------------------
# 3. Comparar nombres
# ------------------------------------------------------------

def comparar_nombres(nombre_moma, nombre_wd):

    n1 = normalizar(nombre_moma)
    n2 = normalizar(nombre_wd)

    if not n1 or not n2:
        return 0, None

    if n1 == n2:
        return 3, "Nombre exacto"

    # Coincidencia parcial de palabras.
    palabras1 = set(n1.split())
    palabras2 = set(n2.split())

    if palabras1 and palabras2:
        interseccion = palabras1.intersection(palabras2)

        similitud = len(interseccion) / max(
            len(palabras1),
            len(palabras2)
        )

        if similitud >= 0.7:
            return 1, "Nombre parcialmente coincidente"

    return 0, None


# ------------------------------------------------------------
# 4. Comparar fechas
# ------------------------------------------------------------

def comparar_fecha(anio_moma, anio_wd, nombre_fecha):

    if anio_moma is None or anio_wd is None:
        return 0, None, None

    if anio_moma == anio_wd:
        return 4, f"{nombre_fecha} coincidente", False

    return -5, f"{nombre_fecha} diferente", True


# ------------------------------------------------------------
# 5. Comparar género
# ------------------------------------------------------------

def comparar_genero(genero_moma, genero_wd):

    g1 = normalizar(genero_moma)
    g2 = normalizar(genero_wd)

    if not g1 or not g2:
        return 0, None, False

    equivalencias = {
        "male": "masculino",
        "man": "masculino",
        "masculino": "masculino",
        "female": "femenino",
        "woman": "femenino",
        "femenino": "femenino"
    }

    g1 = equivalencias.get(g1, g1)
    g2 = equivalencias.get(g2, g2)

    if g1 == g2:
        return 2, "Género coincidente", False

    return -4, "Género diferente", True


# ------------------------------------------------------------
# 6. Comparar nacionalidad y país
# ------------------------------------------------------------

def comparar_pais(nacionalidad_moma, pais_wd):

    n1 = normalizar(nacionalidad_moma)
    n2 = normalizar(pais_wd)

    if not n1 or not n2:
        return 0, None

    # Comparación textual aproximada.
    if n1 in n2 or n2 in n1:
        return 1, "País/nacionalidad coincidente"

    equivalencias = {
        "american": "estados unidos",
        "united states": "estados unidos",
        "british": "reino unido",
        "danish": "dinamarca",
        "swedish": "suecia",
        "spanish": "espana",
        "italian": "italia",
        "french": "francia",
        "german": "alemania"
    }

    n1 = equivalencias.get(n1, n1)
    n2 = equivalencias.get(n2, n2)

    if n1 == n2:
        return 1, "País/nacionalidad coincidente"

    return 0, None


# ------------------------------------------------------------
# 7. Evaluar un candidato
# ------------------------------------------------------------

def evaluar_candidato(row):

    puntuacion = 0
    coincidencias = []
    contradicciones = []

    # Nombre
    puntos, senal = comparar_nombres(
        row["nombre_moma"],
        row["nombre_wikidata"]
    )

    puntuacion += puntos

    if senal:
        coincidencias.append(senal)

    # Nacimiento
    nacimiento_moma = obtener_anio(row["BeginDate"])
    nacimiento_wd = obtener_anio(row["nacimiento"])

    puntos, senal, contradiccion = comparar_fecha(
        nacimiento_moma,
        nacimiento_wd,
        "Nacimiento"
    )

    puntuacion += puntos

    if senal:
        if contradiccion:
            contradicciones.append(senal)
        else:
            coincidencias.append(senal)

    # Fallecimiento
    fallecimiento_moma = obtener_anio(row["EndDate"])
    fallecimiento_wd = obtener_anio(row["fallecimiento"])

    puntos, senal, contradiccion = comparar_fecha(
        fallecimiento_moma,
        fallecimiento_wd,
        "Fallecimiento"
    )

    puntuacion += puntos

    if senal:
        if contradiccion:
            contradicciones.append(senal)
        else:
            coincidencias.append(senal)

    # Género
    puntos, senal, contradiccion = comparar_genero(
        row["Gender"],
        row["genero"]
    )

    puntuacion += puntos

    if senal:
        if contradiccion:
            contradicciones.append(senal)
        else:
            coincidencias.append(senal)

    # Nacionalidad
    puntos, senal = comparar_pais(
        row["Nationality"],
        row["pais"]
    )

    puntuacion += puntos

    if senal:
        coincidencias.append(senal)

    return pd.Series({
        "puntuacion": puntuacion,
        "senales_utilizadas": "; ".join(coincidencias),
        "contradicciones": "; ".join(contradicciones),
        "numero_coincidencias": len(coincidencias),
        "numero_contradicciones": len(contradicciones)
    })


# ------------------------------------------------------------
# 8. Incorporar datos originales de MoMA
# ------------------------------------------------------------

# Recuperamos fechas, género y nacionalidad de MoMA
# usando ConstituentID como identificador.

columnas_moma = [
    "ConstituentID",
    "BeginDate",
    "EndDate",
    "Gender",
    "Nationality"
]

df_moma = df[columnas_moma].drop_duplicates(
    subset=["ConstituentID"]
)

df_evaluacion = df_revision.merge(
    df_moma,
    on="ConstituentID",
    how="left"
)


# ------------------------------------------------------------
# 9. Evaluar todos los candidatos
# ------------------------------------------------------------

resultados = df_evaluacion.apply(
    evaluar_candidato,
    axis=1
)

df_evaluacion = pd.concat(
    [df_evaluacion, resultados],
    axis=1
)


# ------------------------------------------------------------
# 10. Seleccionar el mejor candidato por artista
# ------------------------------------------------------------

# Se ordenan los candidatos por puntuación.
# Solo se considera el mejor candidato de cada artista.
#
# Si hay empate en la puntuación máxima, no se acepta
# automáticamente ninguno: ambos quedan pendientes.

df_evaluacion = df_evaluacion.sort_values(
    by=["ConstituentID", "puntuacion"],
    ascending=[True, False]
)

df_evaluacion["decision"] = "Rechazar"
df_evaluacion["tipo_enlace"] = ""

for constituent_id, grupo in df_evaluacion.groupby(
    "ConstituentID"
):

    max_puntuacion = grupo["puntuacion"].max()

    mejores = grupo[
        grupo["puntuacion"] == max_puntuacion
    ]

    # No aceptar automáticamente si existe empate.
    if len(mejores) > 1:
        df_evaluacion.loc[
            mejores.index,
            "decision"
        ] = "Pendiente de revisión"

        continue

    mejor = mejores.iloc[0]
    idx = mejores.index[0]

    # Reglas para aceptar
    if (
        mejor["puntuacion"] >= 10
        and mejor["numero_coincidencias"] >= 2
        and mejor["numero_contradicciones"] == 0
    ):

        df_evaluacion.loc[idx, "decision"] = "Aceptar"
        df_evaluacion.loc[idx, "tipo_enlace"] = "owl:sameAs"

    # Reglas para dejar pendiente
    elif (
        mejor["numero_contradicciones"] > 0
        or mejor["puntuacion"] > 0
    ):

        df_evaluacion.loc[
            idx,
            "decision"
        ] = "Pendiente de revisión"

    # El resto permanece rechazado.


# ------------------------------------------------------------
# 11. Exportar resultados
# ------------------------------------------------------------

columnas_salida = [
    "ConstituentID",
    "nombre_moma",
    "qid",
    "nombre_wikidata",
    "nacimiento",
    "fallecimiento",
    "pais",
    "genero",
    "puntuacion",
    "senales_utilizadas",
    "contradicciones",
    "decision",
    "tipo_enlace"
]

df_resultado = df_evaluacion[columnas_salida].copy()

df_resultado.to_csv(
    "/content/reconciliacion_automatica.csv",
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 12. Resumen de resultados
# ------------------------------------------------------------

print("Total de candidatos evaluados:", len(df_resultado))

print("\nDecisiones:")
print(df_resultado["decision"].value_counts())

print("\nArtistas aceptados:")
display(
    df_resultado[
        df_resultado["decision"] == "Aceptar"
    ]
)

print("\nArtistas pendientes:")
display(
    df_resultado[
        df_resultado["decision"] == "Pendiente de revisión"
    ]
)

print("\nArchivo generado:")
print("/content/reconciliacion_automatica.csv")


Total de candidatos evaluados: 190

Decisiones:
decision
Rechazar                 95
Aceptar                  50
Pendiente de revisión    45
Name: count, dtype: int64

Artistas aceptados:


,ConstituentID,nombre_moma,qid,nombre_wikidata,nacimiento,fallecimiento,pais,genero,puntuacion,senales_utilizadas,contradicciones,decision,tipo_enlace
0,1,Robert Arneson,Q7341594,Robert Arneson,1930-09-04T00:00:00Z,1992-11-02T00:00:00Z,Estados Unidos,masculino,14,Nombre exacto; Nacimiento coincidente; Falleci...,,Aceptar,owl:sameAs
1,2,Doroteo Arnaiz,Q55996241,Doroteo Arnáiz,1936-01-01T00:00:00Z,NaN,España,masculino,10,Nombre exacto; Nacimiento coincidente; Género ...,,Aceptar,owl:sameAs
7,4,Charles Arnoldi,Q1063584,Charles Arnoldi,1946-04-10T00:00:00Z,NaN,Estados Unidos,masculino,10,Nombre exacto; Nacimiento coincidente; Género ...,,Aceptar,owl:sameAs
12,7,Bill Aron,Q4907949,Bill Aron,1941-01-01T00:00:00Z,NaN,Estados Unidos,masculino,10,Nombre exacto; Nacimiento coincidente; Género ...,,Aceptar,owl:sameAs
19,13,José Arellano Fischer,Q134413342,José Arellano Fischer,1911-01-01T00:00:00Z,1995-01-01T00:00:00Z,NaN,NaN,11,Nombre exacto; Nacimiento coincidente; Falleci...,,Aceptar,owl:sameAs
20,15,Folke Arstrom,Q3747067,Folke Arström,1907-10-15T00:00:00Z,1997-10-15T00:00:00Z,Suecia,masculino,14,Nombre exacto; Nacimiento coincidente; Falleci...,,Aceptar,owl:sameAs
21,16,Cristobal Arteche,Q77353440,Cristobal Arteche,1900-01-01T00:00:00Z,1964-01-01T00:00:00Z,España,masculino,14,Nombre exacto; Nacimiento coincidente; Falleci...,,Aceptar,owl:sameAs
22,19,Richard Artschwager,Q568262,Richard Artschwager,1923-12-26T00:00:00Z,2013-02-09T00:00:00Z,Estados Unidos,masculino,14,Nombre exacto; Nacimiento coincidente; Falleci...,,Aceptar,owl:sameAs
23,21,Ruth Asawa,Q7382874,Ruth Asawa,1926-01-24T00:00:00Z,2013-08-06T00:00:00Z,Estados Unidos,femenino,14,Nombre exacto; Nacimiento coincidente; Falleci...,,Aceptar,owl:sameAs
25,23,Charles Robert Ashbee,Q614071,Charles Robert Ashbee,1863-05-17T00:00:00Z,1942-05-23T00:00:00Z,Reino Unido,masculino,14,Nombre exacto; Nacimiento coincidente; Falleci...,,Aceptar,owl:sameAs



Artistas pendientes:


,ConstituentID,nombre_moma,qid,nombre_wikidata,nacimiento,fallecimiento,pais,genero,puntuacion,senales_utilizadas,contradicciones,decision,tipo_enlace
6,3,Bill Arnold,Q77287962,Q77287962,1941-01-01T00:00:00Z,NaN,Estados Unidos,masculino,7,Nacimiento coincidente; Género coincidente; Pa...,,Pendiente de revisión,
9,5,Per Arnoldi,Q3373273,Per Arnoldi,1941-05-25T00:00:00Z,NaN,Reino de Dinamarca,masculino,9,Nombre exacto; Nacimiento coincidente; Género ...,,Pendiente de revisión,
10,6,Danilo Aroldi,Q55217954,Danilo Aroldi,1925-01-01T00:00:00Z,1989-01-01T00:00:00Z,Italia,masculino,5,Nombre exacto; Nacimiento coincidente; Género ...,Fallecimiento diferente,Pendiente de revisión,
14,9,David Aronson,Q5230870,David Aronson,1923-10-28T00:00:00Z,2015-07-02T00:00:00Z,Estados Unidos,masculino,5,Nombre exacto; Nacimiento coincidente; Género ...,Fallecimiento diferente,Pendiente de revisión,
16,10,Irene Aronson,Q19748568,Irene Aronson,1918-03-08T00:00:00Z,1992-01-01T00:00:00Z,Estados Unidos,femenino,5,Nombre exacto; Nacimiento coincidente; Género ...,Fallecimiento diferente,Pendiente de revisión,
18,12,Jüri Arrak,Q927697,Jüri Arrak,1936-10-24T00:00:00Z,2022-10-16T00:00:00Z,Estonia,masculino,5,Nombre exacto; Nacimiento coincidente; Género ...,Fallecimiento diferente,Pendiente de revisión,
24,22,Isidora Aschheim,Q12403756,Isidor Aschheim,1891-10-14T00:00:00Z,1968-05-19T00:00:00Z,Israel,masculino,-13,País/nacionalidad coincidente,Nacimiento diferente; Fallecimiento diferente;...,Pendiente de revisión,
26,24,Donald Ashcraft,Q55217956,Q55217956,1927-01-01T00:00:00Z,NaN,Estados Unidos,masculino,7,Nacimiento coincidente; Género coincidente; Pa...,,Pendiente de revisión,
27,25,E. M. Ashe,Q19668052,Edmund M. Ashe,1870-06-19T00:00:00Z,1941-01-01T00:00:00Z,Estados Unidos,masculino,2,Fallecimiento coincidente; Género coincidente;...,Nacimiento diferente,Pendiente de revisión,
29,26,Göran Åslin,Q77353556,Göran Åslin,1940-01-01T00:00:00Z,NaN,Suecia,masculino,8,Nombre exacto; Nacimiento coincidente; País/na...,,Pendiente de revisión,



Archivo generado:
/content/reconciliacion_automatica.csv


In [30]:
# Crear los enlaces en el grafo RDF

from rdflib import Graph, URIRef, Literal, Namespace
from rdflib.namespace import OWL

MOMA = Namespace("http://example.org/moma/artist/")
WD = Namespace("http://www.wikidata.org/entity/")
EX = Namespace("http://example.org/moma/evidence/")

g_enlaces = Graph()

g_enlaces.bind("moma", MOMA)
g_enlaces.bind("wd", WD)
g_enlaces.bind("owl", OWL)
g_enlaces.bind("ex", EX)

aceptados = df_resultado[
    df_resultado["decision"] == "Aceptar"
]

for _, artista in aceptados.iterrows():

    iri_moma = URIRef(
        str(MOMA) + str(int(artista["ConstituentID"]))
    )

    iri_wikidata = URIRef(
        str(WD) + str(artista["qid"])
    )

    # Crear correspondencia de identidad
    g_enlaces.add((
        iri_moma,
        OWL.sameAs,
        iri_wikidata
    ))

    # Registrar evidencia
    evidencia = (
        f"Puntuación: {artista['puntuacion']}. "
        f"Coincidencias: {artista['senales_utilizadas']}. "
        f"Contradicciones: {artista['contradicciones']}"
    )

    g_enlaces.add((
        iri_moma,
        EX.evidencia,
        Literal(evidencia)
    ))

# Exportar enlaces
g_enlaces.serialize(
    destination="/content/enlaces.ttl",
    format="turtle"
)

# Incorporar enlaces al grafo original
for triple in g_enlaces:
    g.add(triple)

print("Artistas enlazados:", len(aceptados))
print("Triples generados:", len(g_enlaces))

Artistas enlazados: 50
Triples generados: 100


PARTE CINCO — Evaluación de la reconciliación

In [32]:
# Cargar el archivo original de referencia
import pandas as pd
import numpy as np

# Cargar fuente original de MoMA con los QID de referencia
df_referencia = pd.read_csv(
    "/content/moma-artists-wikidata-qids.txt"
)

print("Artistas en la fuente de referencia:", len(df_referencia))

display(
    df_referencia[
        ["ConstituentID", "DisplayName", "Wiki QID"]
    ].head(10)
)

Artistas en la fuente de referencia: 90


,ConstituentID,DisplayName,Wiki QID
0,1,Robert Arneson,NaN
1,2,Doroteo Arnaiz,NaN
2,3,Bill Arnold,NaN
3,4,Charles Arnoldi,Q1063584
4,5,Per Arnoldi,NaN
5,6,Danilo Aroldi,NaN
6,7,Bill Aron,NaN
7,9,David Aronson,Q5230870
8,10,Irene Aronson,Q19748568
9,11,Jean (Hans) Arp,Q153739


In [33]:
# Comparar los resultados de reconciliación

# ============================================================
# EVALUACIÓN DE LA RECONCILIACIÓN
# ============================================================
#
# VP (Verdadero positivo):
# El QID aceptado por el algoritmo coincide con el QID
# registrado en la fuente original.
#
# FP (Falso positivo):
# El algoritmo aceptó un QID diferente al registrado
# en la fuente original.
#
# FN (Falso negativo):
# La fuente original tiene un QID, pero el algoritmo
# no aceptó ese QID.
#
# Si se acepta un QID incorrecto cuando existe un QID
# de referencia, se contabiliza como FP y FN:
# - FP porque se propuso una entidad equivocada.
# - FN porque no se recuperó la entidad correcta.
#
# Si la fuente original no tiene QID, el caso no se
# puede evaluar y se clasifica como "Sin referencia".
# ============================================================


# ------------------------------------------------------------
# 1. Preparar los QID de referencia
# ------------------------------------------------------------

df_ref = df_referencia[
    ["ConstituentID", "DisplayName", "Wiki QID"]
].copy()

df_ref = df_ref.rename(columns={
    "DisplayName": "nombre_referencia",
    "Wiki QID": "qid_referencia"
})


# Normalizar identificadores
def limpiar_qid(valor):

    if pd.isna(valor):
        return None

    valor = str(valor).strip()

    if valor in ["", "0", "nan", "None"]:
        return None

    return valor


df_ref["qid_referencia"] = df_ref["qid_referencia"].apply(
    limpiar_qid
)


# ------------------------------------------------------------
# 2. Obtener los QID aceptados por nuestro algoritmo
# ------------------------------------------------------------

df_aceptados = df_resultado[
    df_resultado["decision"] == "Aceptar"
][
    ["ConstituentID", "nombre_moma", "qid"]
].copy()

df_aceptados = df_aceptados.rename(columns={
    "qid": "qid_propuesto"
})

df_aceptados["qid_propuesto"] = (
    df_aceptados["qid_propuesto"].apply(limpiar_qid)
)


# ------------------------------------------------------------
# 3. Cruzar ambas fuentes mediante ConstituentID
# ------------------------------------------------------------

df_evaluacion = df_ref.merge(
    df_aceptados,
    on="ConstituentID",
    how="left"
)


# ------------------------------------------------------------
# 4. Clasificar resultados
# ------------------------------------------------------------

def clasificar(row):

    referencia = row["qid_referencia"]
    propuesto = row["qid_propuesto"]

    # No existe QID original para comparar
    if referencia is None:
        return "Sin referencia"

    # El algoritmo no aceptó ningún candidato
    if propuesto is None:
        return "FN"

    # Coincidencia exacta
    if referencia == propuesto:
        return "VP"

    # Se aceptó una entidad diferente
    return "FP + FN"


df_evaluacion["clasificacion"] = df_evaluacion.apply(
    clasificar,
    axis=1
)


# Indicadores individuales para facilitar el conteo
df_evaluacion["VP"] = (
    df_evaluacion["clasificacion"] == "VP"
).astype(int)

df_evaluacion["FP"] = (
    df_evaluacion["clasificacion"].isin(
        ["FP + FN"]
    )
).astype(int)

df_evaluacion["FN"] = (
    df_evaluacion["clasificacion"].isin(
        ["FN", "FP + FN"]
    )
).astype(int)


# ------------------------------------------------------------
# 5. Mostrar resultados detallados
# ------------------------------------------------------------

display(
    df_evaluacion[
        [
            "ConstituentID",
            "nombre_referencia",
            "qid_referencia",
            "qid_propuesto",
            "clasificacion"
        ]
    ]
)


# ------------------------------------------------------------
# 6. Calcular métricas
# ------------------------------------------------------------

VP = df_evaluacion["VP"].sum()
FP = df_evaluacion["FP"].sum()
FN = df_evaluacion["FN"].sum()

precision = VP / (VP + FP) if (VP + FP) > 0 else 0
recall = VP / (VP + FN) if (VP + FN) > 0 else 0

f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall) > 0
    else 0
)

print("\n========== RESULTADOS DE EVALUACIÓN ==========")

print("Verdaderos positivos (VP):", VP)
print("Falsos positivos (FP):", FP)
print("Falsos negativos (FN):", FN)

print("\n========== MÉTRICAS ==========")

print(f"Precisión: {precision:.2%}")
print(f"Recall:    {recall:.2%}")
print(f"F1-score:  {f1:.2%}")


,ConstituentID,nombre_referencia,qid_referencia,qid_propuesto,clasificacion
0,1,Robert Arneson,None,Q7341594,Sin referencia
1,2,Doroteo Arnaiz,None,Q55996241,Sin referencia
2,3,Bill Arnold,None,NaN,Sin referencia
3,4,Charles Arnoldi,Q1063584,Q1063584,VP
4,5,Per Arnoldi,None,NaN,Sin referencia
...,...,...,...,...,...
85,96,Anni Albers,Q86078,Q86078,VP
86,97,Josef Albers,Q170071,Q170071,VP
87,98,Beatrice Albert,None,NaN,Sin referencia
88,99,Joan Albert,None,Q63119722,Sin referencia



========== RESULTADOS DE EVALUACIÓN ==========
Verdaderos positivos (VP): 25
Falsos positivos (FP): 17
Falsos negativos (FN): 17

========== MÉTRICAS ==========
Precisión: 59.52%
Recall:    59.52%
F1-score:  59.52%


In [34]:
# Exportar los resultados
df_evaluacion.to_csv(
    "/content/evaluacion_reconciliacion.csv",
    index=False,
    encoding="utf-8-sig"
)

from google.colab import files

files.download("/content/evaluacion_reconciliacion.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

PARTE SEIS — Consultas sobre datos enlazados

In [35]:
# Preparar los artistas confirmados
import pandas as pd
import requests
from rdflib import Graph, Namespace, URIRef

# Si el grafo g todavía existe, lo reutilizamos.
# Si no existe, cargamos el archivo original de MoMA.
try:
    g
except NameError:
    g = Graph()
    g.parse("/content/artistas.ttl", format="turtle")

# Namespaces del grafo local
MOMA = Namespace("http://example.org/moma/artist/")
ARTIST = Namespace("http://example.org/moma/ontology#")
SCHEMA = Namespace("https://schema.org/")

# Seleccionar exclusivamente verdaderos positivos
df_vp = df_evaluacion[
    df_evaluacion["clasificacion"] == "VP"
].copy()

df_vp = df_vp.drop_duplicates(subset=["ConstituentID"])

df_vp["qid_propuesto"] = df_vp["qid_propuesto"].astype(str)

print("Artistas confirmados:", len(df_vp))
display(df_vp[["ConstituentID", "nombre_referencia", "qid_propuesto"]])

Artistas confirmados: 25


,ConstituentID,nombre_referencia,qid_propuesto
3,4,Charles Arnoldi,Q1063584
15,19,Richard Artschwager,Q568262
16,21,Ruth Asawa,Q7382874
18,23,Charles Robert Ashbee,Q614071
22,27,Erik Gunnar Asplund,Q312070
33,39,James Abbe,Q321301
35,41,Berenice Abbott,Q231861
40,47,Raimund Abraham,Q78819
44,52,Rodolfo Abularach,Q7357131
45,53,Vito Acconci,Q433471


In [36]:
# Recuperar la información local de MoMA
datos_locales = []

for _, artista in df_vp.iterrows():

    iri = URIRef(
        f"http://example.org/moma/artist/{artista['ConstituentID']}"
    )

    datos_locales.append({
        "ConstituentID": artista["ConstituentID"],
        "qid": artista["qid_propuesto"],
        "nombre_moma": g.value(iri, SCHEMA.name),
        "nacionalidad_moma": g.value(iri, ARTIST.nationality),
        "genero_moma": g.value(iri, ARTIST.gender),
        "nacimiento_moma": g.value(iri, ARTIST.beginDate),
        "fallecimiento_moma": g.value(iri, ARTIST.endDate)
    })

df_local = pd.DataFrame(datos_locales)

display(df_local.head())

,ConstituentID,qid,nombre_moma,nacionalidad_moma,genero_moma,nacimiento_moma,fallecimiento_moma
0,4,Q1063584,Charles Arnoldi,American,male,1946,0
1,19,Q568262,Richard Artschwager,American,male,1923,2013
2,21,Q7382874,Ruth Asawa,American,female,1926,2013
3,23,Q614071,Charles Robert Ashbee,British,male,1863,1942
4,27,Q312070,Erik Gunnar Asplund,Swedish,male,1885,1940


In [37]:
# Preparar la conexión con Wikidata
import requests

WIKIDATA_ENDPOINT = "https://query.wikidata.org/sparql"

def ejecutar_consulta_wikidata(query):

    response = requests.get(
        WIKIDATA_ENDPOINT,
        params={
            "query": query,
            "format": "json"
        },
        headers={
            "User-Agent": "WebSemantica-Uniandes/1.0"
        },
        timeout=60
    )

    response.raise_for_status()

    return response.json()

In [38]:
qids = df_vp["qid_propuesto"].dropna().unique().tolist()

valores_qids = " ".join(
    f"wd:{qid}" for qid in qids
)

print("QID que se consultarán:", len(qids))

QID que se consultarán: 25


### Consulta 1

In [39]:
"""
Consulta uno: ocupaciones de los artistas
a) Pregunta que queremos responder

¿Qué profesiones u ocupaciones tienen registradas en Wikidata los artistas del conjunto de MoMA?

b) Información del grafo local:
- Nombre del artista.
- Nacionalidad.
- Identificador de MoMA.
c) Información de Wikidata:

- Ocupaciones registradas mediante la propiedad P106 (occupation).
"""

query_ocupaciones = f"""
SELECT ?artist ?artistLabel
       (GROUP_CONCAT(DISTINCT ?occupationLabel; separator="; ") AS ?ocupaciones)
WHERE {{

    VALUES ?artist {{ {valores_qids} }}

    OPTIONAL {{
        ?artist wdt:P106 ?occupation .
        ?occupation rdfs:label ?occupationLabel .

        FILTER(
            LANG(?occupationLabel) = "es" ||
            LANG(?occupationLabel) = "en"
        )
    }}

    SERVICE wikibase:label {{
        bd:serviceParam wikibase:language "es,en".
    }}
}}
GROUP BY ?artist ?artistLabel
"""

resultado_ocupaciones = ejecutar_consulta_wikidata(
    query_ocupaciones
)

filas = []

for item in resultado_ocupaciones["results"]["bindings"]:

    qid = item["artist"]["value"].split("/")[-1]

    filas.append({
        "qid": qid,
        "ocupaciones_wikidata": item.get(
            "ocupaciones", {}
        ).get("value", "")
    })

df_ocupaciones = pd.DataFrame(filas)

# Combinar información local y externa
resultado_1 = df_local.merge(
    df_ocupaciones,
    on="qid",
    how="left"
)

display(
    resultado_1[
        ["nombre_moma", "nacionalidad_moma", "ocupaciones_wikidata"]
    ]
)

,nombre_moma,nacionalidad_moma,ocupaciones_wikidata
0,Charles Arnoldi,American,artist; artista; painter; pintor; sculptor; es...
1,Richard Artschwager,American,painter; pintor; sculptor; escultor; visual ar...
2,Ruth Asawa,American,sculptor; escultor; lithographer; litógrafo; o...
3,Charles Robert Ashbee,British,architect; arquitecto; author; autor; business...
4,Erik Gunnar Asplund,Swedish,arquitecto; architect
5,James Abbe,American,photographer; fotógrafo; periodista; journalis...
6,Berenice Abbott,American,photographer; fotógrafo; artist; artista; inve...
7,Raimund Abraham,American,arquitecto; architect; university teacher; pro...
8,Rodolfo Abularach,Guatemalan,artist; artista; draftsperson; lithographer; l...
9,Vito Acconci,American,photographer; fotógrafo; arquitecto; architect...


### Consulta 2

In [40]:
"""
Consulta dos: movimientos artísticos
a) Pregunta que queremos responder

¿A qué movimientos o corrientes artísticas están asociados los artistas del conjunto de MoMA?

b) Información del grafo local:
- Nombre del artista.
- Nacionalidad.
- Género.
c) Información de Wikidata:

- Movimiento artístico asociado, utilizando P135 (movement).
"""

query_movimientos = f"""
SELECT ?artist ?artistLabel
       (GROUP_CONCAT(DISTINCT ?movementLabel; separator="; ") AS ?movimientos)
WHERE {{

    VALUES ?artist {{ {valores_qids} }}

    OPTIONAL {{
        ?artist wdt:P135 ?movement .
        ?movement rdfs:label ?movementLabel .

        FILTER(
            LANG(?movementLabel) = "es" ||
            LANG(?movementLabel) = "en"
        )
    }}

    SERVICE wikibase:label {{
        bd:serviceParam wikibase:language "es,en".
    }}
}}
GROUP BY ?artist ?artistLabel
"""

resultado_movimientos = ejecutar_consulta_wikidata(
    query_movimientos
)

filas = []

for item in resultado_movimientos["results"]["bindings"]:

    qid = item["artist"]["value"].split("/")[-1]

    filas.append({
        "qid": qid,
        "movimientos_wikidata": item.get(
            "movimientos", {}
        ).get("value", "")
    })

df_movimientos = pd.DataFrame(filas)

resultado_2 = df_local.merge(
    df_movimientos,
    on="qid",
    how="left"
)

display(
    resultado_2[
        ["nombre_moma", "nacionalidad_moma",
         "genero_moma", "movimientos_wikidata"]
    ]
)

,nombre_moma,nacionalidad_moma,genero_moma,movimientos_wikidata
0,Charles Arnoldi,American,male,
1,Richard Artschwager,American,male,abstract art; arte abstracto
2,Ruth Asawa,American,female,
3,Charles Robert Ashbee,British,male,Arts and Crafts movement; Arts and Crafts
4,Erik Gunnar Asplund,Swedish,male,
5,James Abbe,American,male,
6,Berenice Abbott,American,female,straight photography; fotografía directa
7,Raimund Abraham,American,male,
8,Rodolfo Abularach,Guatemalan,male,
9,Vito Acconci,American,male,contemporary art; arte contemporáneo


### Consulta 3

In [41]:
"""

Consulta tres: lugares de nacimiento
a) Pregunta que queremos responder

¿En qué lugares y países nacieron los artistas de MoMA?

b) Información del grafo local:
- Nombre del artista.
- Año de nacimiento.
- Nacionalidad.
c) Información de Wikidata:
- Lugar de nacimiento mediante P19 (place of birth).
- País del lugar de nacimiento mediante P17 (country).

"""

query_nacimiento = f"""
SELECT ?artist ?artistLabel
       (GROUP_CONCAT(DISTINCT ?placeLabel; separator="; ") AS ?lugares_nacimiento)
       (GROUP_CONCAT(DISTINCT ?countryLabel; separator="; ") AS ?paises_nacimiento)
WHERE {{

    VALUES ?artist {{ {valores_qids} }}

    OPTIONAL {{
        ?artist wdt:P19 ?place .

        ?place rdfs:label ?placeLabel .

        FILTER(
            LANG(?placeLabel) = "es" ||
            LANG(?placeLabel) = "en"
        )

        OPTIONAL {{
            ?place wdt:P17 ?country .
            ?country rdfs:label ?countryLabel .

            FILTER(
                LANG(?countryLabel) = "es" ||
                LANG(?countryLabel) = "en"
            )
        }}
    }}

    SERVICE wikibase:label {{
        bd:serviceParam wikibase:language "es,en".
    }}
}}
GROUP BY ?artist ?artistLabel
"""

resultado_nacimiento = ejecutar_consulta_wikidata(
    query_nacimiento
)

filas = []

for item in resultado_nacimiento["results"]["bindings"]:

    qid = item["artist"]["value"].split("/")[-1]

    filas.append({
        "qid": qid,
        "lugares_nacimiento": item.get(
            "lugares_nacimiento", {}
        ).get("value", ""),
        "paises_nacimiento": item.get(
            "paises_nacimiento", {}
        ).get("value", "")
    })

df_nacimiento = pd.DataFrame(filas)

resultado_3 = df_local.merge(
    df_nacimiento,
    on="qid",
    how="left"
)

display(
    resultado_3[
        ["nombre_moma", "nacimiento_moma",
         "nacionalidad_moma", "lugares_nacimiento",
         "paises_nacimiento"]
    ]
)

,nombre_moma,nacimiento_moma,nacionalidad_moma,lugares_nacimiento,paises_nacimiento
0,Charles Arnoldi,1946,American,Dayton; Dayton,United States; Estados Unidos
1,Richard Artschwager,1923,American,"Washington D. C.; Washington, D.C.",United States; Estados Unidos
2,Ruth Asawa,1926,American,Norwalk; Norwalk,United States; Estados Unidos
3,Charles Robert Ashbee,1863,British,Isleworth; Isleworth; Londres; London,United Kingdom; Reino Unido
4,Erik Gunnar Asplund,1885,Swedish,Estocolmo; Stockholm,Sweden; Suecia
5,James Abbe,1883,American,Alfred; Alfred,United States; Estados Unidos
6,Berenice Abbott,1898,American,Springfield; Springfield,United States; Estados Unidos
7,Raimund Abraham,1933,American,Lienz; Lienz,Austria; Austria
8,Rodolfo Abularach,1933,Guatemalan,Ciudad de Guatemala; Guatemala City,Guatemala; Guatemala
9,Vito Acconci,1940,American,Nueva York; New York City,United States; Estados Unidos
